# 03 · Moral agents: full fine-tuning (arm A) vs LoRA (arm B)

Runtime > Change runtime type > **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import getpass, os
GITHUB_USER = 'YOUR_GITHUB_USERNAME'
REPO = 'YOUR_REPO_NAME'
token = getpass.getpass('GitHub token (leave blank if repo is public): ')
auth = f'{token}@' if token else ''
%cd /content
!rm -rf /content/repo
!git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO}.git /content/repo
%cd /content/repo
del token, auth

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/NLP-Final'
os.environ['SCRATCH_DIR'] = '/content/scratch'

### 1. Build the training data (once)
One file per value plus the control. Check how many qualifying pairs each value has.

In [ ]:
!python src/prepare_agent_data.py

### 2. Smoke test both arms on 64 examples
The full fine-tune is the one that could run out of memory, so this checks it in about a minute.

In [ ]:
!python src/train_agent.py --mode lora --value care --n_train 64 --epochs 1 --tag _smoke --overwrite
!python src/train_agent.py --mode full --value care --n_train 64 --epochs 1 --tag _smoke --overwrite

In [ ]:
# Remove the smoke-test models so they don't use Drive space
!rm -rf "$PROJECT_ROOT"/models/agents/*_smoke "$PROJECT_ROOT"/results/agent_*_smoke.json

### 3. Train the care agent in both arms

In [ ]:
!python src/train_agent.py --mode lora --value care
!python src/train_agent.py --mode full --value care

### 4. Score both care agents on the 500 held-out questions

In [ ]:
!python src/generate_and_score.py --tag lora_care --adapter "$PROJECT_ROOT/models/agents/lora_care" --n 500
!python src/generate_and_score.py --tag full_care --model "$PROJECT_ROOT/models/agents/full_care" --n 500

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 300)
R = os.environ['PROJECT_ROOT'] + '/results'
for tag in ['pt', 'lora_care', 'full_care']:
    df = pd.read_csv(f'{R}/generations_{tag}.csv')
    print(tag, '\n', df['reply'].head(3).tolist(), '\n')